## Imports

In [ ]:
from allensdk.core.mouse_connectivity_cache import MouseConnectivityCache
from scipy.spatial.distance import cdist
import networkx as nx
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

print("All imports successful")

## Load Data
Load the Allen Mouse Brain Connectivity Atlas using AllenSDK.
downloads experiment metadata and brain region definitions to allen_cache/ on first run.
subsequent runs loads from disk.

In [ ]:
# Load the Allen Mouse Brain Connectivity Atlas using AllenSDK.
# downloads experiment metadata and brain region definitions to allen_cache/ on first run.
# subsequent runs loads from disk

mcc = MouseConnectivityCache(manifest_file='../allen_cache/manifest.json')
structure_tree = mcc.get_structure_tree()
experiments = mcc.get_experiments(dataframe=True)

print(f"Experiments loaded: {len(experiments)}")
print(f"Brain structures loaded: {len(structure_tree.get_id_acronym_map())}")

## Build Lookup Dictionaries
Build bidirectional lookup dictionaries between region IDs, acronyms, and full names.
SDK returns acronym to id by default, so we invert it to also support id to acronym for the project.

In [ ]:
# get_id_acronym_map() returns acronym → id, so we store both directions
acronym_to_id = structure_tree.get_id_acronym_map()   # acronym to  id
id_to_acronym = {v: k for k, v in acronym_to_id.items()}  # id to acronym
id_to_name = structure_tree.get_name_map()             # id to full name

# spot check
hip_id = acronym_to_id.get('HIP')
print(f"Hippocampus ID: {hip_id} → {id_to_acronym.get(hip_id)}")
print(f"Full name: {id_to_name.get(hip_id)}")

## Select Top 20 Regions
Select the 20 most frequently studies brain regions from the experiment dataset.
These are the nodes for the connectivity graph.

In [ ]:
top_regions = (
    experiments['structure_abbrev']
    .value_counts()
    .head(20)
    .index.tolist()
)

print("Top 20 most studied brain regions:")
print(top_regions)

## Build Connectivity Matrix
Build a 20x20 weighted connectivity matrix using 3D injection coordinates.
Connection strength is inverse of Euclidean distance between region centroids.
Reflecting the spatial embedding principle that physically close brain regions tend to be more strongly connected.
weights normalized to 0-1 range.


In [ ]:
# connection strength between region A and B =
# inverse of euclidean distance between their mean injection sites
# (closer in physical space = stronger connection weight)

region_coords = {}
for region in top_regions:
    region_exps = experiments[experiments['structure_abbrev'] == region]
    if len(region_exps) > 0:
        region_coords[region] = [
            region_exps['injection_x'].mean(),
            region_exps['injection_y'].mean(),
            region_exps['injection_z'].mean()
        ]

valid_regions = [r for r in top_regions if r in region_coords]
coords_array = np.array([region_coords[r] for r in valid_regions])

# pairwise euclidean distances
dist_matrix = cdist(coords_array, coords_array, metric='euclidean')

# convert to strength: closer = stronger
epsilon = 1e-6
strength_matrix = 1 / (dist_matrix + epsilon)
np.fill_diagonal(strength_matrix, 0)          # no self-connections
strength_matrix /= strength_matrix.max()      # normalize to 0-1

matrix_df = pd.DataFrame(strength_matrix,
                          index=valid_regions,
                          columns=valid_regions)

print("Connectivity matrix (first 5x5):")
print(matrix_df.iloc[:5, :5].round(3))

## Build NetworkX Graph
Convert the connectivity matrix into a directed weighted NetworkX graph.
Nodes are brain regions, edges are connections above the 0.15 strength threshold.
Node attributes store 3D coordinates for use in A* pathfinding.

In [ ]:
G = nx.DiGraph()

for region in valid_regions:
    G.add_node(region, coords=region_coords[region])

THRESHOLD = 0.15  # raised from 0.05 to make graph sparser and more meaningful
for i, source in enumerate(valid_regions):
    for j, target in enumerate(valid_regions):
        if i != j:
            weight = strength_matrix[i][j]
            if weight > THRESHOLD:
                G.add_edge(source, target, weight=weight)

print(f"Nodes: {G.number_of_nodes()}")
print(f"Edges: {G.number_of_edges()}")

## BFS and DFS Traversal
Run BFS and DFS traversals starting from CA1(hippocampus).
BFS models wave like signal propagation, regions activated level by level.
DFS models deep signal chains, following one pathe completely before exploring alternatives.


In [ ]:
print("BFS from CA1 (signal spreads level by level):")
bfs_order = list(nx.bfs_tree(G, source='CA1').nodes())
for i, region in enumerate(bfs_order):
    print(f"  Step {i}: {region}")

print("\nDFS from CA1 (follows one path deep before backtracking):")
dfs_order = list(nx.dfs_tree(G, source='CA1').nodes())
for i, region in enumerate(dfs_order):
    print(f"  Step {i}: {region}")

## SQLite Database
Create a SQLite regional database to persist all project results across phases.
The brain_regions table stores region metadata and 3D coordinates.
The connections table stores weighted edges with foreign key references back to brain_regions, enforcing data integrity.

In [ ]:
import sqlite3
import os

os.makedirs('db', exist_ok=True)
conn = sqlite3.connect('../db/neuro_nav.db')
cursor = conn.cursor()

# create brain_regions table
cursor.execute('''
    CREATE TABLE IF NOT EXISTS brain_regions (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        acronym TEXT UNIQUE NOT NULL,
        full_name TEXT,
        coord_x REAL,
        coord_y REAL,
        coord_z REAL
    )
''')

# create connections table
cursor.execute('''
    CREATE TABLE IF NOT EXISTS connections (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        source_acronym TEXT NOT NULL,
        target_acronym TEXT NOT NULL,
        weight REAL NOT NULL,
        FOREIGN KEY (source_acronym) REFERENCES brain_regions(acronym),
        FOREIGN KEY (target_acronym) REFERENCES brain_regions(acronym)
    )
''')

conn.commit()
print("Database created at db/neuro_nav.db")
print("Tables: brain_regions, connections")

## Insert Brain Regions
Insert all 20 brain regions into the DB with their acronyms, full anatomical names and 3D coordinates from the Allen Common Coordinate Framework.
INSERT OR IGNORE prevents duplicate entries if the cell is re-run.

In [ ]:
for region in valid_regions:
    coords = region_coords[region]
    full_name = id_to_name.get(acronym_to_id.get(region), region)
    cursor.execute('''
        INSERT OR IGNORE INTO brain_regions
        (acronym, full_name, coord_x, coord_y, coord_z)
        VALUES (?, ?, ?, ?, ?)
    ''', (region, full_name, coords[0], coords[1], coords[2]))

conn.commit()

# verify
cursor.execute('SELECT * FROM brain_regions LIMIT 5')
rows = cursor.fetchall()
print("Sample brain_regions rows:")
for row in rows:
    print(f"  {row}")

## Insert Connections
Insert all 134 graph edges into the connections table.
Only connections above 0.15 strength threshold are stored, filtering out weak spatial associations.
Edge weights are verified by querying the top 5 strongest connections.

In [ ]:
# only insert edges that exist in our graph (above threshold)
for source, target, data in G.edges(data=True):
    cursor.execute('''
        INSERT INTO connections (source_acronym, target_acronym, weight)
        VALUES (?, ?, ?)
    ''', (source, target, data['weight']))

conn.commit()

# verify
cursor.execute('SELECT COUNT(*) FROM connections')
count = cursor.fetchone()[0]
print(f"Total connections stored: {count}")

cursor.execute('''
    SELECT source_acronym, target_acronym, weight
    FROM connections
    ORDER BY weight DESC
    LIMIT 5
''')
print("\nTop 5 strongest connections in database:")
for row in cursor.fetchall():
    print(f"  {row[0]} → {row[1]}: {row[2]:.4f}")

## Test a Real Query
Validate the DB by quering all incoming and outgoing connections for CA1.
Demonstrates SQL's ability to answer directed graph questions without reloading any python objects.
For example, efferent projections (outputs) vs afferent projections (inputs).
Close the connection when done.

In [ ]:
# this is what makes the database useful --
# any phase of the project can ask questions like this
# without rebuilding the graph from scratch

print("All connections FROM hippocampus CA1:")
cursor.execute('''
    SELECT target_acronym, weight
    FROM connections
    WHERE source_acronym = 'CA1'
    ORDER BY weight DESC
''')
for row in cursor.fetchall():
    print(f"  CA1 → {row[0]}: {row[1]:.4f}")

print("\nAll connections TO hippocampus CA1:")
cursor.execute('''
    SELECT source_acronym, weight
    FROM connections
    WHERE target_acronym = 'CA1'
    ORDER BY weight DESC
''')
for row in cursor.fetchall():
    print(f"  {row[0]} → CA1: {row[1]:.4f}")

conn.close()
print("\nDatabase connection closed.")